### ASTRADB VectorStore
Go from app idea to production with the AI Platform with Astra DB, the ultra-low latency database made for AI and Langflow, the low-code RAG IDE
https://www.datastax.com/

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

In [7]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-3-small",
                            # dimension=1024,
                            api_key =os.getenv("OPENAI_API_KEY"))
embedding



OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001E393C8E960>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001E393D4B1D0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [9]:
 
from langchain_astradb import AstraDBVectorStore
vector_store = AstraDBVectorStore(
    embedding=embedding,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN, 
    collection_name="rag_poc_collection",
    namespace=None
)
vector_store

In [10]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [11]:
vector_store.add_documents(documents = documents)

['8a6a3ea8e15c49139edbad14ffc1d62a',
 '348cdddfb64c4152946ff831c47eccb5',
 '0e353050e60743d0af15987b2ce604d3',
 '5d4e98fa4eb748d582ed09ad37a4e27a',
 '9e8de4d92dda42228cab99e243105d68',
 '9d58b12e87e346cdbc73c1189aaf6fa0',
 '0c741dc79e074c5984f80f5df080813b',
 '61bbfdcd82304e428d890e9d08406758',
 '44a83a1aaaf24195928dd12b3db0f47d',
 '45090d50dd924d83bf0d7f7531e5a02b']

In [13]:
vector_store.similarity_search("What is the Weather?")
 

[Document(id='348cdddfb64c4152946ff831c47eccb5', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='44a83a1aaaf24195928dd12b3db0f47d', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='45090d50dd924d83bf0d7f7531e5a02b', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='9d58b12e87e346cdbc73c1189aaf6fa0', metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.')]

In [15]:
# Apply the filters

results = vector_store.similarity_search(
    "Langchain provides abstractions to make working with LLSs easy",
    k=3,
    filter ={"source":"tweet"}
)

for res in results:
    print(f'* "{res.page_content}", metadata ={res.metadata} ' )

* "LangGraph is the best framework for building stateful, agentic applications!", metadata ={'source': 'tweet'} 
* "Building an exciting new project with LangChain - come check it out!", metadata ={'source': 'tweet'} 
* "I had chocolate chip pancakes and scrambled eggs for breakfast this morning.", metadata ={'source': 'tweet'} 


In [17]:
# Convert vector to retriver 
retriver = vector_store.as_retriever(
    search_type ="similarity_score_threshold",
    search_kwargs={"k" :1, "score_threshold":0.5},
)

retriver.invoke("Stealing from the bank is a Crime", filter={"source":"news"})

[Document(id='5d4e98fa4eb748d582ed09ad37a4e27a', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]